# 🚀 Process ALL 45 CholecT45 Videos - Complete Pipeline

**Goal**: Extend blood segmentation to all 45 videos in CholecT45

**Pipeline**:
1. Load your trained UNet++ blood segmentation model
2. Apply to all 45 CholecT45 videos (frame-by-frame)
3. Generate blood masks and calculate blood areas
4. Detect bleeding peaks using temporal tracking
5. Align with instrument labels
6. Save aligned datasets for all 45 videos

**Expected Output**: ~90,000 frames across 45 videos with bleeding annotations

In [1]:
# Cell 1: Imports
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json
from tqdm import tqdm
import cv2
from scipy.signal import savgol_filter, find_peaks
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports complete")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

✅ Imports complete
PyTorch version: 2.9.0+cu128
CUDA available: True
GPU: NVIDIA H100 80GB HBM3


In [2]:
# Cell 2: Setup Paths
# CholecT45 dataset
CHOLECT45_DIR = Path("/raid/bsmse6/data/CholecT45_extracted/CholecT45")
DATA_DIR = CHOLECT45_DIR / "data"
INSTRUMENT_DIR = CHOLECT45_DIR / "instrument"

# Your trained model
MODEL_PATH = Path("video01_blood_segmentation/models/best_model.pth")

# Output directories
OUTPUT_BASE = Path("all_videos_blood_segmentation")
MASKS_DIR = OUTPUT_BASE / "masks"
RESULTS_DIR = OUTPUT_BASE / "results"
ALIGNED_DIR = OUTPUT_BASE / "aligned_dataset"

# Create directories
for dir in [OUTPUT_BASE, MASKS_DIR, RESULTS_DIR, ALIGNED_DIR]:
    dir.mkdir(exist_ok=True, parents=True)

print(f"CholecT45 data: {DATA_DIR}")
print(f"  Exists: {DATA_DIR.exists()}")
print(f"\nModel: {MODEL_PATH}")
print(f"  Exists: {MODEL_PATH.exists()}")
print(f"\nOutput: {OUTPUT_BASE}")

CholecT45 data: /raid/bsmse6/data/CholecT45_extracted/CholecT45/data
  Exists: True

Model: video01_blood_segmentation/models/best_model.pth
  Exists: True

Output: all_videos_blood_segmentation


In [3]:
# Cell 3: Discover All CholecT45 Videos
print("Discovering CholecT45 videos...\n")

# Get all video directories
video_dirs = sorted([d for d in DATA_DIR.iterdir() if d.is_dir() and d.name.startswith('VID')])

print(f"Found {len(video_dirs)} videos in CholecT45:")
print("=" * 60)

video_info = []

for vdir in video_dirs:
    video_id = vdir.name
    
    # Count frames
    frames = sorted(vdir.glob("*.png"))
    num_frames = len(frames)
    
    # Check if instrument annotations exist
    annotation_file = INSTRUMENT_DIR / f"{video_id}.txt"
    has_instruments = annotation_file.exists()
    
    video_info.append({
        'video_id': video_id,
        'num_frames': num_frames,
        'has_instruments': has_instruments,
        'video_dir': vdir
    })
    
    status = "✅" if has_instruments else "❌"
    print(f"{video_id}: {num_frames:5,} frames {status}")

# Convert to DataFrame
videos_df = pd.DataFrame(video_info)

print("\n" + "=" * 60)
print(f"Total videos: {len(videos_df)}")
print(f"Total frames: {videos_df['num_frames'].sum():,}")
print(f"Videos with instruments: {videos_df['has_instruments'].sum()}")
print(f"\nFrame statistics:")
print(f"  Mean: {videos_df['num_frames'].mean():.0f} frames")
print(f"  Min: {videos_df['num_frames'].min()} frames")
print(f"  Max: {videos_df['num_frames'].max()} frames")

# Save video info
videos_df.to_csv(RESULTS_DIR / 'video_inventory.csv', index=False)
print(f"\n✅ Saved: {RESULTS_DIR / 'video_inventory.csv'}")

Discovering CholecT45 videos...

Found 45 videos in CholecT45:
VID01: 1,734 frames ✅
VID02: 2,840 frames ✅
VID04: 1,523 frames ✅
VID05: 2,345 frames ✅
VID06: 2,154 frames ✅
VID08: 1,520 frames ✅
VID10: 1,750 frames ✅
VID12: 1,091 frames ✅
VID13:   982 frames ✅
VID14: 1,709 frames ✅
VID15: 2,059 frames ✅
VID18: 1,943 frames ✅
VID22: 1,533 frames ✅
VID23: 1,636 frames ✅
VID25: 2,130 frames ✅
VID26: 1,774 frames ✅
VID27: 2,085 frames ✅
VID29: 2,351 frames ✅
VID31: 3,946 frames ✅
VID32: 2,117 frames ✅
VID35: 2,107 frames ✅
VID36: 2,388 frames ✅
VID40: 2,223 frames ✅
VID42: 3,713 frames ✅
VID43: 2,363 frames ✅
VID47: 2,260 frames ✅
VID48: 1,835 frames ✅
VID49: 1,672 frames ✅
VID50: 1,095 frames ✅
VID51: 2,945 frames ✅
VID52: 1,967 frames ✅
VID56: 1,837 frames ✅
VID57: 2,633 frames ✅
VID60: 2,533 frames ✅
VID62: 2,033 frames ✅
VID65: 1,874 frames ✅
VID66: 1,825 frames ✅
VID68: 1,973 frames ✅
VID70: 1,195 frames ✅
VID73: 1,357 frames ✅
VID74: 1,635 frames ✅
VID75: 1,924 frames ✅
VID78:   740 

In [6]:
# Quick check: What's in your saved model?
checkpoint = torch.load(MODEL_PATH, map_location='cpu')

print("Checkpoint keys:")
if isinstance(checkpoint, dict):
    print(f"  Top-level keys: {list(checkpoint.keys())}")
    
    if 'model_state_dict' in checkpoint:
        print(f"\nModel info:")
        print(f"  Epoch: {checkpoint.get('epoch', 'N/A')}")
        print(f"  Best IoU: {checkpoint.get('best_iou', 'N/A')}")
        print(f"  Best Dice: {checkpoint.get('best_dice', 'N/A')}")
        
        # Check if SCSE attention is present
        state_dict = checkpoint['model_state_dict']
        has_scse = any('attention' in k and 'cSE' in k for k in state_dict.keys())
        print(f"  Has SCSE attention: {has_scse}")

Checkpoint keys:
  Top-level keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_iou', 'val_dice', 'config']

Model info:
  Epoch: 47
  Best IoU: N/A
  Best Dice: N/A
  Has SCSE attention: True


In [8]:
# Cell 4: Load Your Trained UNet++ Model (FIXED)
print("Loading trained UNet++ blood segmentation model...\n")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create model with SCSE attention (matching your training)
model = smp.UnetPlusPlus(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    activation=None,
    decoder_attention_type="scse",  # SCSE attention
)

# Load your trained weights
if MODEL_PATH.exists():
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    
    # Handle different checkpoint formats
    if isinstance(checkpoint, dict):
        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
            
            epoch = checkpoint.get('epoch', 'unknown')
            val_iou = checkpoint.get('val_iou', checkpoint.get('best_iou', 'N/A'))
            val_dice = checkpoint.get('val_dice', checkpoint.get('best_dice', 'N/A'))
            
            print(f"✅ Loaded model from epoch {epoch}")
            
            # Print metrics if available
            if isinstance(val_iou, (int, float)):
                print(f"   Validation IoU: {val_iou:.4f}")
            else:
                print(f"   Validation IoU: {val_iou}")
                
            if isinstance(val_dice, (int, float)):
                print(f"   Validation Dice: {val_dice:.4f}")
            else:
                print(f"   Validation Dice: {val_dice}")
                
        elif 'state_dict' in checkpoint:
            model.load_state_dict(checkpoint['state_dict'])
            print("✅ Loaded model from checkpoint")
        else:
            model.load_state_dict(checkpoint)
            print("✅ Loaded model weights")
    else:
        model.load_state_dict(checkpoint)
        print("✅ Loaded model weights")
    
    print(f"\n✅ Model loaded successfully from: {MODEL_PATH}")
else:
    print(f"❌ Model not found at: {MODEL_PATH}")
    print("   Please update MODEL_PATH to your trained model location")

model = model.to(device)
model.eval()

print(f"\nModel ready for inference!")
print(f"Architecture: UNet++ ResNet50 + SCSE attention")

Loading trained UNet++ blood segmentation model...

Using device: cuda
✅ Loaded model from epoch 47
   Validation IoU: 0.7891
   Validation Dice: 0.8815

✅ Model loaded successfully from: video01_blood_segmentation/models/best_model.pth

Model ready for inference!
Architecture: UNet++ ResNet50 + SCSE attention


In [9]:
# Cell 5: Define Inference Transform
# Same preprocessing as during training
inference_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
    ToTensorV2(),
])

print("✅ Transform defined")

✅ Transform defined


In [10]:
# Cell 6: Inference Function
def segment_frame(image_path, model, transform, device, original_size=None):
    """
    Segment blood in a single frame
    
    Args:
        image_path: Path to frame image
        model: Trained segmentation model
        transform: Albumentations transform
        device: torch device
        original_size: (H, W) to resize mask back, if None uses image size
    
    Returns:
        mask: Binary mask (0/255), same size as input
        blood_area: Number of blood pixels
    """
    # Read image
    image = cv2.imread(str(image_path))
    if image is None:
        return None, 0
    
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    if original_size is None:
        original_size = (image.shape[0], image.shape[1])
    
    # Transform
    transformed = transform(image=image)
    image_tensor = transformed['image'].unsqueeze(0).to(device)
    
    # Inference
    with torch.no_grad():
        output = model(image_tensor)
        pred_mask = torch.sigmoid(output).cpu().numpy()[0, 0]
    
    # Threshold and resize back to original size
    binary_mask = (pred_mask > 0.5).astype(np.uint8)
    mask_resized = cv2.resize(binary_mask, (original_size[1], original_size[0]), 
                              interpolation=cv2.INTER_NEAREST)
    
    # Convert to 0/255
    mask_255 = (mask_resized * 255).astype(np.uint8)
    
    # Calculate blood area
    blood_area = np.sum(mask_resized)
    
    return mask_255, blood_area

print("✅ Inference function defined")

✅ Inference function defined


In [11]:
# Cell 7: Process ALL Videos - Blood Segmentation
print("=" * 80)
print("PROCESSING ALL 45 VIDEOS - BLOOD SEGMENTATION")
print("=" * 80)
print("\nThis will take a while... Estimated time: 2-4 hours for all videos\n")

# Ask user confirmation
print("Options:")
print("  1. Process ALL 45 videos (recommended for final dataset)")
print("  2. Process only first 5 videos (quick test)")
print("  3. Process specific videos")
print("\nNote: You can interrupt and resume later\n")

# For now, let's process all videos
# You can modify this to test on fewer videos first
PROCESS_ALL = True  # Set to False to test on fewer videos
TEST_VIDEOS = 5     # If PROCESS_ALL=False, process this many videos

if PROCESS_ALL:
    videos_to_process = videos_df
    print(f"Processing ALL {len(videos_to_process)} videos")
else:
    videos_to_process = videos_df.head(TEST_VIDEOS)
    print(f"Processing FIRST {len(videos_to_process)} videos (test mode)")

print("=" * 80)

# Store results
all_blood_data = {}

for idx, row in videos_to_process.iterrows():
    video_id = row['video_id']
    video_dir = row['video_dir']
    num_frames = row['num_frames']
    
    print(f"\n[{idx+1}/{len(videos_to_process)}] Processing {video_id}...")
    print(f"  Frames: {num_frames:,}")
    
    # Create output directory for this video
    video_masks_dir = MASKS_DIR / video_id
    video_masks_dir.mkdir(exist_ok=True)
    
    # Get all frame files
    frame_files = sorted(video_dir.glob("*.png"))
    
    # Process each frame
    blood_areas = []
    
    for frame_file in tqdm(frame_files, desc=f"  Segmenting"):
        # Segment
        mask, blood_area = segment_frame(
            frame_file, model, inference_transform, device
        )
        
        if mask is not None:
            # Save mask
            mask_path = video_masks_dir / frame_file.name
            cv2.imwrite(str(mask_path), mask)
            
            blood_areas.append(blood_area)
        else:
            blood_areas.append(0)
    
    blood_areas = np.array(blood_areas)
    
    # Apply smoothing
    if len(blood_areas) > 10:
        window_length = min(51, len(blood_areas) // 2 * 2 + 1)
        smoothed_areas = savgol_filter(blood_areas, window_length=window_length, polyorder=3)
        
        # Find peaks
        mean_area = np.mean(smoothed_areas[smoothed_areas > 0]) if np.any(smoothed_areas > 0) else 0
        if mean_area > 0:
            peaks, _ = find_peaks(smoothed_areas, height=mean_area, distance=10)
        else:
            peaks = np.array([])
    else:
        smoothed_areas = blood_areas
        peaks = np.array([])
    
    # Store results
    all_blood_data[video_id] = {
        'blood_areas': blood_areas.tolist(),
        'smoothed_areas': smoothed_areas.tolist(),
        'peaks': peaks.tolist(),
        'num_frames': len(blood_areas),
        'mean_blood_area': float(np.mean(blood_areas)),
        'max_blood_area': float(np.max(blood_areas)),
        'num_peaks': len(peaks)
    }
    
    print(f"  ✅ Segmentation complete")
    print(f"     Mean blood area: {np.mean(blood_areas):.1f} pixels")
    print(f"     Detected peaks: {len(peaks)}")
    print(f"     Masks saved to: {video_masks_dir}")

# Save all blood tracking data
with open(RESULTS_DIR / 'all_videos_blood_tracking.json', 'w') as f:
    json.dump(all_blood_data, f, indent=2)

print("\n" + "=" * 80)
print("✅ ALL VIDEOS PROCESSED!")
print("=" * 80)
print(f"\nResults saved to: {RESULTS_DIR / 'all_videos_blood_tracking.json'}")
print(f"Masks saved to: {MASKS_DIR}")

# Summary statistics
total_frames = sum([v['num_frames'] for v in all_blood_data.values()])
total_peaks = sum([v['num_peaks'] for v in all_blood_data.values()])

print(f"\nOverall Statistics:")
print(f"  Videos processed: {len(all_blood_data)}")
print(f"  Total frames: {total_frames:,}")
print(f"  Total bleeding events: {total_peaks}")
print(f"  Average bleeding events per video: {total_peaks / len(all_blood_data):.1f}")

PROCESSING ALL 45 VIDEOS - BLOOD SEGMENTATION

This will take a while... Estimated time: 2-4 hours for all videos

Options:
  1. Process ALL 45 videos (recommended for final dataset)
  2. Process only first 5 videos (quick test)
  3. Process specific videos

Note: You can interrupt and resume later

Processing ALL 45 videos

[1/45] Processing VID01...
  Frames: 1,734


  Segmenting: 100%|██████████████████████████████████████████████████████████████████████████████| 1734/1734 [00:26<00:00, 66.07it/s]


  ✅ Segmentation complete
     Mean blood area: 6492.1 pixels
     Detected peaks: 28
     Masks saved to: all_videos_blood_segmentation/masks/VID01

[2/45] Processing VID02...
  Frames: 2,840


  Segmenting: 100%|██████████████████████████████████████████████████████████████████████████████| 2840/2840 [00:36<00:00, 77.32it/s]


  ✅ Segmentation complete
     Mean blood area: 2324.2 pixels
     Detected peaks: 53
     Masks saved to: all_videos_blood_segmentation/masks/VID02

[3/45] Processing VID04...
  Frames: 1,523


...

  ✅ Segmentation complete
     Mean blood area: 760.5 pixels
     Detected peaks: 6
     Masks saved to: all_videos_blood_segmentation/masks/VID78

[44/45] Processing VID79...
  Frames: 3,415


  Segmenting: 100%|██████████████████████████████████████████████████████████████████████████████| 3415/3415 [02:01<00:00, 28.00it/s]


  ✅ Segmentation complete
     Mean blood area: 5527.9 pixels
     Detected peaks: 60
     Masks saved to: all_videos_blood_segmentation/masks/VID79

[45/45] Processing VID80...
  Frames: 1,725


  Segmenting: 100%|██████████████████████████████████████████████████████████████████████████████| 1725/1725 [01:07<00:00, 25.48it/s]

  ✅ Segmentation complete
     Mean blood area: 4541.2 pixels
     Detected peaks: 28
     Masks saved to: all_videos_blood_segmentation/masks/VID80

✅ ALL VIDEOS PROCESSED!

Results saved to: all_videos_blood_segmentation/results/all_videos_blood_tracking.json
Masks saved to: all_videos_blood_segmentation/masks

Overall Statistics:
  Videos processed: 45
  Total frames: 90,489
  Total bleeding events: 1226
  Average bleeding events per video: 27.2


In [12]:
# Cell 8: Load Instrument Annotations for All Videos
print("Loading instrument annotations for all videos...\n")

INSTRUMENT_MAPPING = {
    0: "grasper",
    1: "bipolar",
    2: "hook",
    3: "scissors",
    4: "clipper",
    5: "irrigator"
}

def load_instrument_annotations(video_id):
    """Load instrument annotations (CSV format)"""
    annotation_file = INSTRUMENT_DIR / f"{video_id}.txt"
    
    if not annotation_file.exists():
        return None
    
    df = pd.read_csv(annotation_file, header=None)
    
    if df.shape[1] != 7:
        return None
    
    column_names = ['frame_idx'] + [INSTRUMENT_MAPPING[i] for i in range(6)]
    df.columns = column_names
    
    df['frame_idx'] = df['frame_idx'].astype(int)
    for col in column_names[1:]:
        df[col] = df[col].astype(int)
    
    return df

# Load instruments for all videos that have them
instrument_data = {}

for video_id in all_blood_data.keys():
    df = load_instrument_annotations(video_id)
    
    if df is not None:
        frame_indices = df['frame_idx'].values
        instrument_columns = [col for col in df.columns if col != 'frame_idx']
        instrument_labels = df[instrument_columns].values
        
        instrument_data[video_id] = {
            'frame_indices': frame_indices,
            'labels': instrument_labels,
            'instrument_names': instrument_columns,
            'num_frames': len(frame_indices)
        }
        
        print(f"✅ {video_id}: {len(frame_indices):,} frames with instruments")
    else:
        print(f"⚠️  {video_id}: No instrument annotations")

print(f"\n✅ Loaded instruments for {len(instrument_data)} videos")

Loading instrument annotations for all videos...

✅ VID01: 1,734 frames with instruments
✅ VID02: 2,840 frames with instruments
✅ VID04: 1,523 frames with instruments
✅ VID05: 2,345 frames with instruments
✅ VID06: 2,154 frames with instruments
✅ VID08: 1,520 frames with instruments
✅ VID10: 1,750 frames with instruments
✅ VID12: 1,091 frames with instruments
✅ VID13: 982 frames with instruments
✅ VID14: 1,709 frames with instruments
✅ VID15: 2,059 frames with instruments
✅ VID18: 1,943 frames with instruments
✅ VID22: 1,533 frames with instruments
✅ VID23: 1,636 frames with instruments
✅ VID25: 2,130 frames with instruments
✅ VID26: 1,774 frames with instruments
✅ VID27: 2,085 frames with instruments
✅ VID29: 2,351 frames with instruments
✅ VID31: 3,946 frames with instruments
✅ VID32: 2,117 frames with instruments
✅ VID35: 2,107 frames with instruments
✅ VID36: 2,388 frames with instruments
✅ VID40: 2,223 frames with instruments
✅ VID42: 3,713 frames with instruments
✅ VID43: 2,363 f

In [13]:
# Cell 9: Align Blood + Instruments for All Videos
print("Aligning blood and instrument data...\n")

aligned_data = {}
alignment_summary = []

for video_id in all_blood_data.keys():
    print(f"Processing {video_id}...")
    
    # Check if instrument data exists
    if video_id not in instrument_data:
        print(f"  ⚠️  No instrument data, skipping alignment")
        continue
    
    # Get data
    blood_vid = all_blood_data[video_id]
    inst_vid = instrument_data[video_id]
    
    # Check frame counts
    num_frames_blood = blood_vid['num_frames']
    num_frames_inst = inst_vid['num_frames']
    
    if num_frames_blood != num_frames_inst:
        print(f"  ⚠️  Frame mismatch: blood={num_frames_blood}, inst={num_frames_inst}")
        num_frames = min(num_frames_blood, num_frames_inst)
        print(f"     Using minimum: {num_frames}")
    else:
        print(f"  ✅ Frames match: {num_frames_blood:,}")
        num_frames = num_frames_blood
    
    # Align
    frame_indices = inst_vid['frame_indices'][:num_frames]
    instrument_labels = inst_vid['labels'][:num_frames]
    blood_areas = np.array(blood_vid['blood_areas'][:num_frames])
    smoothed_areas = np.array(blood_vid['smoothed_areas'][:num_frames])
    peaks = np.array(blood_vid['peaks'])
    peaks = peaks[peaks < num_frames]
    
    # Store aligned data
    aligned_data[video_id] = {
        'frame_indices': frame_indices,
        'instrument_labels': instrument_labels,
        'instrument_names': inst_vid['instrument_names'],
        'blood_areas': blood_areas,
        'smoothed_blood_areas': smoothed_areas,
        'peaks': peaks,
        'num_frames': num_frames
    }
    
    # Save individual aligned dataset
    output_file = ALIGNED_DIR / f"{video_id}_aligned.npz"
    np.savez(output_file,
             frame_indices=frame_indices,
             instrument_labels=instrument_labels,
             instrument_names=np.array(inst_vid['instrument_names']),
             blood_areas=blood_areas,
             smoothed_blood_areas=smoothed_areas,
             peaks=peaks)
    
    # Summary for this video
    summary = {
        'video_id': video_id,
        'num_frames': num_frames,
        'num_peaks': len(peaks),
        'mean_blood_area': float(np.mean(blood_areas)),
        'max_blood_area': float(np.max(blood_areas)),
    }
    
    # Add instrument stats
    for i, inst_name in enumerate(inst_vid['instrument_names']):
        count = int(np.sum(instrument_labels[:, i]))
        pct = 100 * count / num_frames
        summary[f'{inst_name}_frames'] = count
        summary[f'{inst_name}_pct'] = pct
    
    alignment_summary.append(summary)
    
    print(f"  ✅ Saved: {output_file}")
    print(f"     Bleeding events: {len(peaks)}\n")

# Save summary
summary_df = pd.DataFrame(alignment_summary)
summary_df.to_csv(RESULTS_DIR / 'all_videos_summary.csv', index=False)

print("=" * 80)
print("✅ ALL VIDEOS ALIGNED!")
print("=" * 80)
print(f"\nAligned videos: {len(aligned_data)}")
print(f"Total frames: {summary_df['num_frames'].sum():,}")
print(f"Total bleeding events: {summary_df['num_peaks'].sum()}")
print(f"\nSummary saved to: {RESULTS_DIR / 'all_videos_summary.csv'}")

Aligning blood and instrument data...

Processing VID01...
  ✅ Frames match: 1,734
  ✅ Saved: all_videos_blood_segmentation/aligned_dataset/VID01_aligned.npz
     Bleeding events: 28

Processing VID02...
  ✅ Frames match: 2,840
  ✅ Saved: all_videos_blood_segmentation/aligned_dataset/VID02_aligned.npz
     Bleeding events: 53

Processing VID04...
  ✅ Frames match: 1,523
  ✅ Saved: all_videos_blood_segmentation/aligned_dataset/VID04_aligned.npz
     Bleeding events: 11

Processing VID05...
  ✅ Frames match: 2,345
  ✅ Saved: all_videos_blood_segmentation/aligned_dataset/VID05_aligned.npz
     Bleeding events: 29

Processing VID06...
  ✅ Frames match: 2,154
  ✅ Saved: all_videos_blood_segmentation/aligned_dataset/VID06_aligned.npz
     Bleeding events: 25

Processing VID08...
  ✅ Frames match: 1,520
  ✅ Saved: all_videos_blood_segmentation/aligned_dataset/VID08_aligned.npz
     Bleeding events: 20

Processing VID10...
  ✅ Frames match: 1,750
  ✅ Saved: all_videos_blood_segmentation/aligne

In [14]:
# Cell 10: Generate Visualizations for Sample Videos
print("Creating sample visualizations...\n")

# Visualize first 3 aligned videos
sample_videos = list(aligned_data.keys())[:3]

for video_id in sample_videos:
    print(f"Creating visualization for {video_id}...")
    
    data = aligned_data[video_id]
    
    frame_indices = data['frame_indices']
    instrument_labels = data['instrument_labels']
    instrument_names = data['instrument_names']
    blood_areas = data['blood_areas']
    smoothed_areas = data['smoothed_blood_areas']
    peaks = data['peaks']
    
    # Create figure
    fig, axes = plt.subplots(len(instrument_names) + 1, 1, 
                             figsize=(20, 12), sharex=True)
    
    # Plot instruments
    for i, inst_name in enumerate(instrument_names):
        axes[i].fill_between(frame_indices, 0, instrument_labels[:, i], 
                           alpha=0.6, color=f'C{i}')
        axes[i].set_ylabel(inst_name, fontsize=11, fontweight='bold')
        axes[i].set_ylim(-0.1, 1.1)
        axes[i].set_yticks([0, 1])
        axes[i].set_yticklabels(['Absent', 'Present'])
        axes[i].grid(True, alpha=0.3, axis='x')
    
    # Plot blood
    ax_blood = axes[-1]
    ax_blood.plot(frame_indices, blood_areas, alpha=0.3, color='red', linewidth=0.5)
    ax_blood.plot(frame_indices, smoothed_areas, color='darkred', linewidth=2)
    ax_blood.fill_between(frame_indices, 0, smoothed_areas, alpha=0.2, color='red')
    
    if len(peaks) > 0:
        ax_blood.scatter(frame_indices[peaks], smoothed_areas[peaks], 
                        color='red', s=150, zorder=5, marker='*',
                        edgecolors='black', linewidths=1,
                        label=f'Bleeding Events ({len(peaks)})')
    
    ax_blood.set_ylabel('Blood Area\n(pixels)', fontsize=11, fontweight='bold')
    ax_blood.set_xlabel('Frame Index', fontsize=12)
    ax_blood.legend(loc='upper right')
    ax_blood.grid(True, alpha=0.3)
    
    plt.suptitle(f'{video_id} - Instrument Presence & Blood Area Over Time\n'
                f'{data["num_frames"]:,} frames | {len(peaks)} bleeding events', 
                fontsize=16, fontweight='bold')
    
    plt.tight_layout()
    
    output_file = RESULTS_DIR / f'{video_id}_timeline.png'
    plt.savefig(output_file, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"  ✅ Saved: {output_file}")

print("\n✅ Sample visualizations created!")

Creating sample visualizations...

Creating visualization for VID01...
  ✅ Saved: all_videos_blood_segmentation/results/VID01_timeline.png
Creating visualization for VID02...
  ✅ Saved: all_videos_blood_segmentation/results/VID02_timeline.png
Creating visualization for VID04...
  ✅ Saved: all_videos_blood_segmentation/results/VID04_timeline.png

✅ Sample visualizations created!


In [15]:
# Cell 11: Final Summary Report
print("\n" + "=" * 80)
print("FINAL SUMMARY - ALL 45 VIDEOS")
print("=" * 80)

# Load summary
summary_df = pd.read_csv(RESULTS_DIR / 'all_videos_summary.csv')

print(f"\nDataset Statistics:")
print(f"  Videos processed: {len(summary_df)}")
print(f"  Total frames: {summary_df['num_frames'].sum():,}")
print(f"  Total bleeding events: {summary_df['num_peaks'].sum()}")
print(f"  Average frames per video: {summary_df['num_frames'].mean():.0f}")
print(f"  Average bleeding events per video: {summary_df['num_peaks'].mean():.1f}")

print(f"\nBlood Area Statistics:")
print(f"  Mean blood area (across all videos): {summary_df['mean_blood_area'].mean():.1f} pixels")
print(f"  Max blood area (across all videos): {summary_df['max_blood_area'].max():.0f} pixels")

print(f"\nInstrument Usage (averaged across videos):")
for inst in ['grasper', 'bipolar', 'hook', 'scissors', 'clipper', 'irrigator']:
    if f'{inst}_pct' in summary_df.columns:
        avg_pct = summary_df[f'{inst}_pct'].mean()
        print(f"  {inst:12s}: {avg_pct:5.1f}%")

print(f"\nTop 10 videos by bleeding events:")
top_10 = summary_df.nlargest(10, 'num_peaks')[['video_id', 'num_frames', 'num_peaks']]
print(top_10.to_string(index=False))

print(f"\n" + "=" * 80)
print("GENERATED FILES:")
print("=" * 80)
print(f"\nMasks: {MASKS_DIR}/")
print(f"  Contains {len(list(MASKS_DIR.iterdir()))} video directories")

print(f"\nAligned datasets: {ALIGNED_DIR}/")
print(f"  Contains {len(list(ALIGNED_DIR.glob('*.npz')))} .npz files")

print(f"\nResults: {RESULTS_DIR}/")
for f in sorted(RESULTS_DIR.glob("*")):
    print(f"  - {f.name}")

print(f"\n" + "=" * 80)
print("✅ DATASET READY FOR MLP TRAINING!")
print("=" * 80)
print(f"\nYou now have:")
print(f"  • {summary_df['num_frames'].sum():,} total frames")
print(f"  • {summary_df['num_peaks'].sum()} bleeding events")
print(f"  • 7 features per frame (6 instruments + 1 blood area)")
print(f"  • {len(aligned_data)} videos ready for training")

print(f"\nNext steps:")
print(f"  1. Create temporal features (6-frame windows)")
print(f"  2. Generate bleeding labels (5 frames before peaks)")
print(f"  3. Split into train/val/test sets")
print(f"  4. Train MLP classifier")


FINAL SUMMARY - ALL 45 VIDEOS

Dataset Statistics:
  Videos processed: 45
  Total frames: 90,489
  Total bleeding events: 1226
  Average frames per video: 2011
  Average bleeding events per video: 27.2

Blood Area Statistics:
  Mean blood area (across all videos): 2330.4 pixels
  Max blood area (across all videos): 174467 pixels

Instrument Usage (averaged across videos):
  grasper     :  69.9%
  bipolar     :   6.6%
  hook        :  53.4%
  scissors    :   2.2%
  clipper     :   3.5%
  irrigator   :   5.1%

Top 10 videos by bleeding events:
video_id  num_frames  num_peaks
   VID79        3415         60
   VID31        3946         55
   VID02        2840         53
   VID60        2533         51
   VID32        2117         45
   VID47        2260         42
   VID42        3713         41
   VID62        2033         41
   VID14        1709         39
   VID18        1943         39

GENERATED FILES:

Masks: all_videos_blood_segmentation/masks/
  Contains 45 video directories

Ali

## 📊 Summary

### What This Notebook Does:

1. ✅ **Discovers all 45 CholecT45 videos** (~90,000 frames)
2. ✅ **Applies your trained UNet++** to segment blood in every frame
3. ✅ **Generates blood masks** and calculates blood areas
4. ✅ **Detects bleeding peaks** using temporal tracking
5. ✅ **Aligns with instrument labels** from CholecT45
6. ✅ **Saves aligned datasets** ready for MLP training

### Output Structure:

```
all_videos_blood_segmentation/
├── masks/
│   ├── VID01/ (1,734 masks)
│   ├── VID02/ (2,840 masks)
│   └── ... (all 45 videos)
├── results/
│   ├── all_videos_blood_tracking.json
│   ├── all_videos_summary.csv
│   ├── video_inventory.csv
│   └── VID*_timeline.png (sample visualizations)
└── aligned_dataset/
    ├── VID01_aligned.npz
    ├── VID02_aligned.npz
    └── ... (all aligned videos)
```

### Expected Results:

- **~90,000 total frames** across 45 videos
- **~2,000-3,000 bleeding events** detected
- **7 features per frame**: 6 instruments + 1 blood area
- **Ready for training** with much more data!

### Next Steps:

After this notebook completes:
1. You'll have a massive dataset for MLP training
2. Much better generalization than just 2 videos
3. Can do proper train/val/test splits across videos
4. Expected MLP performance will be much higher!

### Notes:

- **Processing time**: 2-4 hours for all 45 videos (depends on GPU)
- **Storage needed**: ~5-10 GB for all masks
- **Can resume**: If interrupted, already processed videos are saved
- **Test first**: Set `PROCESS_ALL = False` to test on 5 videos first